# T1.3 — Whisper WER floor check

Verify that `openai/whisper-small` meets the Phase 3 lower bound:
clean WER < 10%, noisy WER < 25% (pink noise at SNR 10 dB).

If noisy WER > 40%, the ASR -> tactic pipeline is too lossy and we
either drop noisy inputs from the eval or upgrade to whisper-medium.

In [ ]:
# --- VishGuard setup cell (copy from notebooks/00_colabSetup.ipynb) ---
import os, sys, subprocess
from pathlib import Path
REPO_URL = os.environ.get('VISHGUARD_REPO_URL', 'https://github.com/ben-blake/vishguard.git')
DRIVE_SRC = '/content/drive/MyDrive/vishguard'
REPO_DIR = Path('/content/vishguard')
ON_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
def sh(cmd, check=True):
    print('$', cmd); subprocess.run(cmd, shell=True, check=check)
if ON_COLAB:
    try:
        from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    except Exception as e: print('Drive mount skipped:', e)
    if REPO_URL:
        if REPO_DIR.exists(): sh(f'cd {REPO_DIR} && git pull --ff-only')
        else: sh(f'git clone {REPO_URL} {REPO_DIR}')
    elif Path(DRIVE_SRC).exists():
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        sh(f'rsync -a --delete --exclude .venv --exclude __pycache__ {DRIVE_SRC}/ {REPO_DIR}/')
    else:
        raise RuntimeError('Set VISHGUARD_REPO_URL or copy the repo to MyDrive/vishguard/')
    sh(f'pip install -q -e {REPO_DIR}')
    sh(f'pip install -q -r {REPO_DIR}/requirements.txt')
    src = str(REPO_DIR / 'src')
    if src not in sys.path: sys.path.insert(0, src)
import vishguard; print('vishguard', vishguard.__version__)

In [ ]:
import numpy as np, torch, librosa
from datasets import load_dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import wer

WHISPER_ID = 'openai/whisper-small'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Use model directly — transformers 5.x pipeline has a num_frames regression
# with the dict-input code path on some Colab runtimes.
processor = WhisperProcessor.from_pretrained(WHISPER_ID)
model = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_ID, torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32
).to(device)
model.eval()
print('loaded:', WHISPER_ID, '  device:', device)

In [ ]:
# 5 clean clips with reference text.
ds = load_dataset('openslr/librispeech_asr', 'clean', split='test', streaming=True,
                  trust_remote_code=True)
clips = []
for i, ex in enumerate(ds):
    if i >= 5: break
    audio = ex['audio']
    samples = audio['array'].astype(np.float32)
    sr = audio['sampling_rate']
    if sr != 16_000:
        samples = librosa.resample(samples, orig_sr=sr, target_sr=16_000)
    clips.append({'samples': samples, 'text': ex['text']})
print(f'clean clips: {len(clips)}  avg_dur={np.mean([len(c["samples"])/16000 for c in clips]):.1f}s')

In [ ]:
def add_pink_noise(samples: np.ndarray, snr_db: float = 10.0) -> np.ndarray:
    # 1/f pink noise via random spectrum shaping.
    n = len(samples)
    spectrum = np.fft.rfft(np.random.randn(n).astype(np.float32))
    freqs = np.fft.rfftfreq(n)
    freqs[0] = freqs[1]
    spectrum /= np.sqrt(freqs)
    noise = np.fft.irfft(spectrum, n=n).astype(np.float32)
    s_pow = float(np.mean(samples ** 2))
    n_pow = float(np.mean(noise ** 2))
    target_n_pow = s_pow / (10 ** (snr_db / 10))
    noise *= np.sqrt(target_n_pow / max(n_pow, 1e-12))
    return samples + noise

noisy_clips = [{'samples': add_pink_noise(c['samples'], 10.0), 'text': c['text']} for c in clips]
print('noisy clips ready')

In [ ]:
import re

def _normalize(text: str) -> str:
    # Strip punctuation Whisper adds but LibriSpeech refs omit.
    return re.sub(r'[^\w\s]', '', text).lower().strip()

def transcribe(clip):
    feats = processor(
        clip['samples'], sampling_rate=16_000,
        return_tensors='pt', return_attention_mask=True,
    )
    input_features = feats.input_features.to(device=device, dtype=model.dtype)
    attn_mask = feats.attention_mask.to(device)
    with torch.no_grad():
        ids = model.generate(input_features, attention_mask=attn_mask)
    return processor.batch_decode(ids, skip_special_tokens=True)[0]

def wer_for(subset):
    refs_raw  = [c['text'].lower() for c in subset]
    hyps_raw  = [transcribe(c).lower() for c in subset]
    refs_norm = [_normalize(r) for r in refs_raw]
    hyps_norm = [_normalize(h) for h in hyps_raw]
    raw_wer  = wer(refs_raw,  hyps_raw)
    norm_wer = wer(refs_norm, hyps_norm)
    pairs = list(zip(refs_raw, hyps_raw))
    return raw_wer, norm_wer, pairs

clean_raw,  clean_norm,  clean_pairs = wer_for(clips)
noisy_raw,  noisy_norm,  noisy_pairs = wer_for(noisy_clips)
print(f'clean WER  raw={clean_raw:.3f}  normalized={clean_norm:.3f}  (bar < 0.10)')
print(f'noisy WER  raw={noisy_raw:.3f}  normalized={noisy_norm:.3f}  (bar < 0.25)')
print('acceptance: clean_norm < 0.10, noisy_norm < 0.25 (escalate if noisy_norm > 0.40)')

In [ ]:
# Spot-check: print the pair with the worst alignment per subset.
from jiwer import wer as _wer
def worst(pairs):
    return max(pairs, key=lambda p: _wer(p[0], p[1]))
print('--- worst clean ---')
print('REF:', worst(clean_pairs)[0])
print('HYP:', worst(clean_pairs)[1])
print('--- worst noisy ---')
print('REF:', worst(noisy_pairs)[0])
print('HYP:', worst(noisy_pairs)[1])

## Acceptance checklist

- [x] clean WER (normalized) < 0.10 — **0.013** (1.3%)
- [x] noisy WER (normalized) < 0.25 — **0.032** (3.2%)
- [x] Spot-check worst-pair output looks plausible

**T1.3 results (Colab T4 run 2026-04-21):**

| Condition | Raw WER | Normalized WER | Bar | Result |
| --- | --- | --- | --- | --- |
| clean | 0.127 | **0.013** | < 0.10 | PASS |
| noisy (SNR 10 dB) | 0.146 | **0.032** | < 0.25 | PASS |

Raw→normalized gap (0.127→0.013 clean) confirms raw score was almost
entirely Whisper punctuation insertions against punct-free LibriSpeech refs.
Actual word-level accuracy is excellent.

**Worst-case spot-check:**
- Clean: 'opened before them' → 'opened for them' — single word substitution
- Noisy: 'concord' → 'concorde' — one character, fully intelligible

T1.3 **PASSED**. whisper-small is sufficient; no upgrade to whisper-medium needed.
Use normalized WER in all Phase 3 eval scripts.